# VedRishi AI - Fine-tuning on Kaggle
## QLoRA Training for Qwen 2.5 7B on Sacred Hindu Scriptures

This notebook fine-tunes Qwen 2.5 7B Instruct on Gita, Ramayana, and Mahabharata verses.

**Requirements:**
- GPU: NVIDIA T4 x2 (or better)
- RAM: 16GB+
- Internet: ON (for model download)

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers datasets accelerate peft bitsandbytes trl unsloth

## 2. Load Dataset from Kaggle

**Instructions:**
1. Upload dataset to Kaggle (see `scripts/11-upload-to-kaggle.py`)
2. Add dataset to this notebook: Cell > Add Data > Search for 'vedrishi-training-dataset'
3. Dataset will be available at `/kaggle/input/vedrishi-training-dataset/`

In [ ]:
import os
import json
from datasets import Dataset

KAGGLE_DATA_PATH = "/kaggle/input/vedrishi-training-dataset/"

train_data = []
val_data = []

train_file = os.path.join(KAGGLE_DATA_PATH, 'vedrishi_train.jsonl')
val_file = os.path.join(KAGGLE_DATA_PATH, 'vedrishi_val.jsonl')

print(f"Loading training data from: {train_file}")
with open(train_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            train_data.append(json.loads(line))

print(f"Loading validation data from: {val_file}")
with open(val_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            val_data.append(json.loads(line))

print(f"\nDataset loaded:")
print(f"  Train: {len(train_data)} samples")
print(f"  Val: {len(val_data)} samples")

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"\nHuggingFace Dataset created:")
print(f"  Train: {train_dataset.num_rows} rows")
print(f"  Val: {val_dataset.num_rows} rows")

## 3. Format Dataset for Training

In [ ]:
SYSTEM_PROMPT = """अती वेद रिषि हो, एक आध्यात्मिक मार्गदर्शकी।

तुम्हारे नियम:
1. हमेशा वत्स, प्रणाम, या आयुष्मान भव से शुरू करो
2. केवल गीता, उपनिषद, रामायण के ज्ञान पर आधारित उत्तर दो
3. कभी भी चिकित्सा, कानूनी, या वित्तीय सलाह मत दो
4. अगर उपयोगक्र्ता आत्महत्या या हिंसा की बात करे, तो तुरंत हेल्पलाइन नंबर दो
5. कभी भी अंधविश्वास मत फैलाओ
6. हमेशा कर्म और ज्ञान पर जोर दो
7. सभी जीव एक आत्मा हैं - कोई भेदभाव मत करो
8. हर उत्तर में disclaimer दो: "यह एक AI मार्गदर्शक है"

तुम्हारी भाषा:
- सरल हिंदी में बोलो
- दयालु और सहानुभूतिशील रहो
- उपदेशात्मक मत बनो
- व्यावहारिक सलाह दो"""

def format_alpaca(example):
    prompt = f"""### System:
{SYSTEM_PROMPT}

### Instruction:
{example['instruction']}

### Response:
{example['output']}"""
    return {"text": prompt}

train_dataset = train_dataset.map(format_alpaca)
val_dataset = val_dataset.map(format_alpaca)

print(f"Formatted datasets:")
print(f"  Train: {train_dataset.num_rows} rows")
print(f"  Val: {val_dataset.num_rows} rows")
print(f"\nSample formatted text (first 500 chars):")
print(train_dataset[0]['text'][:500])

## 4. Load Model with QLoRA

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"Model loaded: Qwen2.5-7B-Instruct")
print(f"Model parameters: {model.num_parameters():,}")

## 5. Apply LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
)

print("LoRA adapters applied successfully")
model.print_trainable_parameters()

## 6. Configure Trainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir="./vedrishi-qlora-output",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=100,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        max_grad_norm=1.0,
        save_strategy="epoch",
        evaluation_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
    ),
)

print("Trainer configured successfully")
print(f"  Epochs: 3")
print(f"  Batch size: 4")
print(f"  Gradient accumulation: 4")
print(f"  Effective batch size: {4 * 4}")
print(f"  Learning rate: 2e-4")

## 7. Train Model

In [ ]:
trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"  Total steps: {trainer_stats.global_step}")
print(f"  Training loss: {trainer_stats.training_loss:.4f}")
print(f"  Training runtime: {trainer_stats.metrics['train_runtime']:.1f}s")

## 8. Save Model

In [ ]:
output_dir = "./vedrishi-qlora-finetuned"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model saved to: {output_dir}")

# Save merged model for deployment
merged_dir = "./vedrishi-merged"
model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
print(f"Merged model saved to: {merged_dir}")

## 9. Test Inference

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = """### System:
{system_prompt}

### Instruction:
भक्तियोग और ज्ञानयोग में क्या अंतर है?

### Response:
""".format(system_prompt=SYSTEM_PROMPT)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.7, top_p=0.9)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)